# Code Blue Prediction Model & Critical Care Optimization Platform

End-to-end Snowflake ML pipeline that predicts the likelihood of a Code Blue (cardiopulmonary arrest) event for hospitalized patients from demographics, flowsheet vitals and laboratory values. It supports early intervention, patient safety, ICU capacity planning and proactive clinical decision-making. Everything runs on warehouse compute (DEMO_WH) in DEMO_DB.CLINIC, with the Feature Store in DEMO_DB.FEATURES.

**Table of contents**
1. Training dataset creation (PATIENT_CODEBLUE, 1,000 synthetic encounters, last 3 months)
2. Feature engineering (Snowpark) + Feature Store (FV_CODE_BLUE_PREDICTION v V1) + point-in-time training set
3. AutoML-style training & evaluation - XGBoost, LightGBM, RandomForest, LogisticRegression, GradientBoosting - tracked in CODE_BLUE_EXPERIMENT
4. Model Registry deployment - CODEBLUE_MODEL version V1_0 with metrics, params, feature definitions, explainability
5. Production inference - CODEBLUE_SCORING_RESULTS
6. Final business dataset - CODEBLUE_PREDICTION_RESULTS

_Assumptions: version names must be SQL identifiers, so suggested v1.0 is registered as V1_0; the default model version is treated as the latest approved version._

In [ ]:
import json, warnings
import numpy as np, pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
warnings.filterwarnings('ignore')

session = get_active_session()
DB, SCHEMA, WH = 'DEMO_DB', 'CLINIC', 'DEMO_WH'
FEATURES = 'FEATURES'                      # Feature Store schema
EXPERIMENT = 'CODE_BLUE_EXPERIMENT'
MODEL_NAME, MODEL_VERSION = 'CODEBLUE_MODEL', 'V1_0'
FQ_MODEL = f'{DB}.{SCHEMA}.{MODEL_NAME}'
MON = f'{DB}.{SCHEMA}.CODEBLUE_MODEL_MONITOR'   # reserved name for a future model monitor
FV_NAME, FV_VERSION = 'FV_CODE_BLUE_PREDICTION', 'V1'
SEED = 42
for s in [f'USE WAREHOUSE {WH}', f'USE DATABASE {DB}', f'USE SCHEMA {SCHEMA}']:
    session.sql(s).collect()
print(session.get_current_database(), session.get_current_schema(), session.get_current_warehouse())

## Phase 1 - Training dataset creation
Generates 1,000 synthetic encounters (admissions over the last 90 days). A latent acuity score drives correlated vitals and labs (high acuity: higher RR/pulse/BNP/BUN/neutrophils, lower SBP/SpO2/albumin/calcium). CODEBLUE is drawn from a clinically motivated logistic model (~6-9% prevalence). DISCHARGE_DATE is always populated and lengthens with acuity and Code Blue (recent admissions may have a projected discharge). About 3% of labs/vitals are set missing to exercise imputation. Patient, Flowsheet and Lab tables are joined on PRIMARYMRN.

In [ ]:
rng = np.random.default_rng(SEED)
N = 1000
now = pd.Timestamp.now().floor('h')
mrn = [f'MRN{1000000 + i}' for i in range(N)]
age = np.clip(rng.normal(62, 17, N), 18, 98).round().astype(int)
gender = rng.choice(['Male', 'Female'], N, p=[0.49, 0.51])
height = np.where(gender == 'Male', rng.normal(176, 7, N), rng.normal(162, 6.5, N)).round(1)
weight = np.clip(np.where(gender == 'Male', rng.normal(195, 40, N), rng.normal(165, 38, N)), 90, 450).round(1)
birth = [(now - pd.Timedelta(days=int(a * 365.25 + rng.integers(0, 365)))).normalize() for a in age]
entry = now - pd.to_timedelta(rng.integers(0, 90 * 24, N), unit='h')

acuity = rng.normal(0, 1, N) + 0.03 * (age - 62)
resp = np.clip(rng.normal(17, 3, N) + 2.5 * acuity, 8, 40).round()
sbp = np.clip(rng.normal(125, 16, N) - 9 * acuity, 60, 210).round()
dbp = np.clip(sbp * 0.62 + rng.normal(0, 7, N), 35, 130).round()
temp = np.clip(rng.normal(37.0, 0.5, N) + 0.3 * acuity, 34.5, 41).round(1)
spo2 = np.clip(rng.normal(96.5, 2, N) - 1.8 * acuity, 70, 100).round()
o2 = np.clip(np.where(spo2 < 93, rng.normal(4, 2, N), rng.exponential(0.6, N)), 0, 15).round(1)
pulse = np.clip(rng.normal(82, 13, N) + 9 * acuity, 40, 180).round()
lymph = np.clip(rng.lognormal(0.3, 0.4, N) - 0.25 * acuity, 0.1, 6).round(2)
neut = np.clip(rng.lognormal(1.5, 0.4, N) + 1.2 * acuity, 0.5, 30).round(2)
alb = np.clip(rng.normal(3.8, 0.5, N) - 0.3 * acuity, 1.5, 5.2).round(1)
bcult = (rng.random(N) < 1 / (1 + np.exp(-(-2.6 + 0.9 * acuity)))).astype(int)
bnp = np.clip(rng.lognormal(4.6 + 0.6 * acuity, 0.9, N), 5, 5000).round()
bun = np.clip(rng.lognormal(2.8 + 0.25 * acuity, 0.4, N), 4, 150).round()
ca = np.clip(rng.normal(9.2, 0.5, N) - 0.2 * acuity, 6.5, 11.5).round(1)

logit = (-4.3 + 1.2 * acuity + 0.05 * (resp - 17) - 0.02 * (sbp - 125) - 0.12 * (spo2 - 96.5)
         + 0.015 * (pulse - 82) + 0.7 * bcult + 0.35 * np.log(bnp / 100) - 0.5 * (alb - 3.8))
codeblue = (rng.random(N) < 1 / (1 + np.exp(-logit))).astype(int)
los_days = np.clip(rng.gamma(2.0, 1.5 + 1.2 * np.maximum(acuity, 0), N) + 5 * codeblue + 0.5, 0.5, 45)
disch = entry + pd.to_timedelta((los_days * 24).round(), unit='h')

def with_missing(a, frac=0.03):
    a = a.astype(float).copy(); a[rng.random(N) < frac] = np.nan; return a

patient = pd.DataFrame({'PRIMARYMRN': mrn, 'BIRTHDATE': birth, 'GENDER': gender, 'AGE': age,
    'HEIGHT_CMS': with_missing(height), 'WEIGHT_LBS': with_missing(weight),
    'ENTRY_DATE': entry, 'DISCHARGE_DATE': disch, 'CODEBLUE': codeblue})
flowsheet = pd.DataFrame({'PRIMARYMRN': mrn, 'RESPIRATION': with_missing(resp), 'SYSTOLIC_BP': with_missing(sbp),
    'DIASTOLIC_BP': with_missing(dbp), 'TEMPERATURE': with_missing(temp), 'PULSE_OXIMETRY': with_missing(spo2),
    'O2_FLOW_RATE': with_missing(o2), 'PULSE': with_missing(pulse)})
lab = pd.DataFrame({'PRIMARYMRN': mrn, 'ABS_LYMPHOCYTE': with_missing(lymph), 'ABS_NEUTROPHIL': with_missing(neut),
    'ALBUMIN': with_missing(alb), 'BLOOD_CULTURE': bcult, 'BNP': with_missing(bnp), 'BUN': with_missing(bun),
    'CALCIUM': with_missing(ca)})

for name, d in [('SRC_PATIENT', patient), ('SRC_FLOWSHEET', flowsheet), ('SRC_VITAL_LAB', lab)]:
    session.write_pandas(d, name, auto_create_table=True, overwrite=True, use_logical_type=True)

session.sql('''CREATE OR REPLACE TABLE PATIENT_CODEBLUE AS
SELECT p.PRIMARYMRN, p.BIRTHDATE::DATE AS BIRTHDATE, p.GENDER, p.AGE, p.HEIGHT_CMS, p.WEIGHT_LBS,
       p.ENTRY_DATE::TIMESTAMP_NTZ AS ENTRY_DATE, p.DISCHARGE_DATE::TIMESTAMP_NTZ AS DISCHARGE_DATE,
       f.RESPIRATION, f.SYSTOLIC_BP, f.DIASTOLIC_BP, f.TEMPERATURE, f.PULSE_OXIMETRY, f.O2_FLOW_RATE, f.PULSE,
       l.ABS_LYMPHOCYTE, l.ABS_NEUTROPHIL, l.ALBUMIN, l.BLOOD_CULTURE, l.BNP, l.BUN, l.CALCIUM,
       p.CODEBLUE
FROM SRC_PATIENT p JOIN SRC_FLOWSHEET f USING (PRIMARYMRN) JOIN SRC_VITAL_LAB l USING (PRIMARYMRN)''').collect()
src = session.table('PATIENT_CODEBLUE')
print('rows:', src.count())
src.group_by('CODEBLUE').count().show()

### Phase 1 Deliverables
| Deliverable | Where |
|---|---|
| PATIENT_CODEBLUE table | `DEMO_DB.CLINIC.PATIENT_CODEBLUE` (created above) |
| Data profiling summary | `profile_summary` (cell below) |
| Dataset statistics & schema documentation | `patient_schema` (SQL) + `schema_doc` (column dictionary) |

In [ ]:
raw_pdf = src.to_pandas()
num_profile_cols = raw_pdf.select_dtypes('number').columns
profile_summary = pd.DataFrame({
    'dtype': raw_pdf.dtypes.astype(str),
    'non_null': raw_pdf.notna().sum(),
    'null_pct': (raw_pdf.isna().mean() * 100).round(2),
    'distinct': raw_pdf.nunique(),
}).join(raw_pdf[num_profile_cols].describe().T[['mean', 'std', 'min', '50%', 'max']].rename(columns={'50%': 'median'}).round(2))
print(f'rows: {len(raw_pdf)} | columns: {raw_pdf.shape[1]} | CODEBLUE rate: {raw_pdf.CODEBLUE.mean():.1%} '
      f'({int(raw_pdf.CODEBLUE.sum())} positives) | entry window: {raw_pdf.ENTRY_DATE.min()} -> {raw_pdf.ENTRY_DATE.max()}')
profile_summary

In [ ]:
%%sql -r patient_schema
SELECT ORDINAL_POSITION, COLUMN_NAME, DATA_TYPE, IS_NULLABLE, NUMERIC_PRECISION, NUMERIC_SCALE
FROM DEMO_DB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'CLINIC' AND TABLE_NAME = 'PATIENT_CODEBLUE'
ORDER BY ORDINAL_POSITION;

In [ ]:
col_desc = {
    'PRIMARYMRN': ('Identifier', 'Patient medical record number (entity key)', '-'),
    'BIRTHDATE': ('Demographic', 'Date of birth', 'date'), 'GENDER': ('Demographic', 'Male / Female', '-'),
    'AGE': ('Demographic', 'Age at admission', 'years'), 'HEIGHT_CMS': ('Demographic', 'Height', 'cm'),
    'WEIGHT_LBS': ('Demographic', 'Weight', 'lb'), 'ENTRY_DATE': ('Encounter', 'Admission timestamp', 'timestamp'),
    'DISCHARGE_DATE': ('Encounter', 'Discharge timestamp', 'timestamp'),
    'RESPIRATION': ('Vital', 'Respiratory rate', 'breaths/min'), 'SYSTOLIC_BP': ('Vital', 'Systolic blood pressure', 'mmHg'),
    'DIASTOLIC_BP': ('Vital', 'Diastolic blood pressure', 'mmHg'), 'TEMPERATURE': ('Vital', 'Body temperature', 'deg C'),
    'PULSE_OXIMETRY': ('Vital', 'Oxygen saturation SpO2', '%'), 'O2_FLOW_RATE': ('Vital', 'Supplemental oxygen flow', 'L/min'),
    'PULSE': ('Vital', 'Heart rate', 'bpm'), 'ABS_LYMPHOCYTE': ('Lab', 'Absolute lymphocyte count', '10^3/uL'),
    'ABS_NEUTROPHIL': ('Lab', 'Absolute neutrophil count', '10^3/uL'), 'ALBUMIN': ('Lab', 'Serum albumin', 'g/dL'),
    'BLOOD_CULTURE': ('Lab', 'Blood culture positive (1) / negative (0)', 'flag'), 'BNP': ('Lab', 'B-type natriuretic peptide', 'pg/mL'),
    'BUN': ('Lab', 'Blood urea nitrogen', 'mg/dL'), 'CALCIUM': ('Lab', 'Serum calcium', 'mg/dL'),
    'CODEBLUE': ('Target', 'Code Blue event during encounter (1 = yes)', 'flag'),
}
schema_doc = pd.DataFrame([(c, *v) for c, v in col_desc.items()], columns=['COLUMN', 'CATEGORY', 'DESCRIPTION', 'UNIT'])
schema_doc = schema_doc.merge(profile_summary[['dtype', 'null_pct', 'min', 'median', 'max']], left_on='COLUMN', right_index=True, how='left')
schema_doc

## Phase 2 - Feature engineering & Feature Store
Snowpark pipeline (pushed down as SQL, so training and inference share identical logic via the feature view): median imputation (medians computed once and baked into the view), GENDER encoded with sklearn LabelEncoder mapping, derived clinical features (BMI, MAP, shock index, neutrophil-lymphocyte ratio, hypoxia flag), date features (LOS days, admission duration hours, day of week, month). Scaling: tree models are scale-invariant; StandardScaler is applied inside the LogisticRegression pipeline so it ships with the model. Note: LOS/discharge-based features are included per requirements but are only known at discharge (potential leakage for real-time use).

Feature view FV_CODE_BLUE_PREDICTION version V1 (physical object FV_CODE_BLUE_PREDICTION$V1 = FV_CODE_BLUE_PREDICTION_V1) on entity PATIENT (PRIMARYMRN), timestamp ENCOUNTER_TS. Training set is built with point-in-time ASOF join via generate_training_set.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

src = session.table(f'{DB}.{SCHEMA}.PATIENT_CODEBLUE')   # fully qualified: Feature Store views live in another schema
le = LabelEncoder().fit(src.select('GENDER').distinct().to_pandas()['GENDER'].dropna())
gender_map = {g: int(i) for i, g in enumerate(le.classes_)}
vitals = ['RESPIRATION', 'SYSTOLIC_BP', 'DIASTOLIC_BP', 'TEMPERATURE', 'PULSE_OXIMETRY', 'O2_FLOW_RATE', 'PULSE']
labs = ['ABS_LYMPHOCYTE', 'ABS_NEUTROPHIL', 'ALBUMIN', 'BLOOD_CULTURE', 'BNP', 'BUN', 'CALCIUM']
num_cols = ['AGE', 'HEIGHT_CMS', 'WEIGHT_LBS'] + vitals + labs
medians = src.select([F.median(c).alias(c) for c in num_cols]).to_pandas().iloc[0].to_dict()

gexpr = F.lit(-1)
for k, v in gender_map.items():
    gexpr = F.when(F.col('GENDER') == k, F.lit(v)).otherwise(gexpr)
d = src
for c in num_cols:
    d = d.with_column(c, F.coalesce(F.col(c).cast('double'), F.lit(float(medians[c]))))
feat_df = d.select(
    'PRIMARYMRN', F.col('ENTRY_DATE').alias('ENCOUNTER_TS'),
    *[F.col(c) for c in num_cols], gexpr.alias('GENDER_ENC'),
    (F.col('WEIGHT_LBS') * 0.453592 / F.pow(F.col('HEIGHT_CMS') / 100, 2)).alias('BMI'),
    ((F.col('SYSTOLIC_BP') + 2 * F.col('DIASTOLIC_BP')) / 3).alias('MEAN_ARTERIAL_PRESSURE'),
    (F.col('PULSE') / F.col('SYSTOLIC_BP')).alias('SHOCK_INDEX'),
    (F.col('ABS_NEUTROPHIL') / F.col('ABS_LYMPHOCYTE')).alias('NEUTROPHIL_LYMPH_RATIO'),
    F.iff(F.col('PULSE_OXIMETRY') < 92, 1, 0).alias('HYPOXIA_FLAG'),
    (F.datediff('hour', F.col('ENTRY_DATE'), F.col('DISCHARGE_DATE')) / 24.0).alias('LOS_DAYS'),
    F.datediff('hour', F.col('ENTRY_DATE'), F.col('DISCHARGE_DATE')).alias('ADMISSION_DURATION_HRS'),
    F.dayofweek('ENTRY_DATE').alias('ENTRY_DOW'), F.month('ENTRY_DATE').alias('ENTRY_MONTH'))
feat_df = feat_df.select('PRIMARYMRN', 'ENCOUNTER_TS', *[F.col(c).cast('double').alias(c) for c in feat_df.columns if c not in ('PRIMARYMRN', 'ENCOUNTER_TS')])

fs = FeatureStore(session=session, database=DB, name=FEATURES, default_warehouse=WH, creation_mode=CreationMode.CREATE_IF_NOT_EXIST)
patient_entity = Entity(name='PATIENT', join_keys=['PRIMARYMRN'], desc='Hospitalized patient encounter (PRIMARYMRN)')
try:
    fs.register_entity(patient_entity)
except Exception as e:
    print('entity exists:', str(e)[:80])
fv = FeatureView(name=FV_NAME, entities=[patient_entity], feature_df=feat_df, timestamp_col='ENCOUNTER_TS',
                 desc='Code Blue predictors - demographics, vitals, labs, derived clinical and date features')
gender_desc = 'LabelEncoder ' + ', '.join(f'{k}={v}' for k, v in gender_map.items())
fv = fv.attach_feature_desc({'BMI': 'Body mass index kg per m2', 'SHOCK_INDEX': 'Pulse divided by systolic BP',
    'MEAN_ARTERIAL_PRESSURE': 'SBP plus 2 x DBP over 3', 'NEUTROPHIL_LYMPH_RATIO': 'ANC divided by ALC',
    'LOS_DAYS': 'Length of stay in days', 'GENDER_ENC': gender_desc})
fv = fs.register_feature_view(feature_view=fv, version=FV_VERSION, overwrite=True)

spine = src.select('PRIMARYMRN', F.col('ENTRY_DATE').alias('ENCOUNTER_TS'), 'CODEBLUE')
try:
    train_sdf = fs.generate_training_set(spine_df=spine, features=[fv], spine_timestamp_col='ENCOUNTER_TS', spine_label_cols=['CODEBLUE'])
except TypeError:
    train_sdf = fs.generate_training_set(spine, [fv], 'ENCOUNTER_TS', ['CODEBLUE'])
train_pdf = train_sdf.to_pandas()
FEATURE_COLS = [c for c in train_pdf.columns if c not in ('PRIMARYMRN', 'ENCOUNTER_TS', 'CODEBLUE')]
print('training set:', train_pdf.shape, '| positives:', int(train_pdf.CODEBLUE.sum()))
print('features:', FEATURE_COLS)

### Phase 2 Deliverables
| Deliverable | Where |
|---|---|
| Reusable Snowpark Python pipeline | `build_codeblue_features()` (cell below), validated against the registered feature view logic |
| Feature transformation documentation | `feature_transform_doc` |
| Feature list used for training | `training_feature_list` (from `FEATURE_COLS`) |

In [ ]:
def build_codeblue_features(df, medians: dict, gender_map: dict, num_cols: list = num_cols):
    """Snowpark (SQL pushdown) feature pipeline for Code Blue scoring.
    df: Snowpark DataFrame with the PATIENT_CODEBLUE schema. medians/gender_map: training-time artifacts."""
    g = F.lit(-1)
    for k, v in gender_map.items():
        g = F.when(F.col('GENDER') == k, F.lit(v)).otherwise(g)
    for c in num_cols:
        df = df.with_column(c, F.coalesce(F.col(c).cast('double'), F.lit(float(medians[c]))))
    out = df.select(
        'PRIMARYMRN', F.col('ENTRY_DATE').alias('ENCOUNTER_TS'), *[F.col(c) for c in num_cols], g.alias('GENDER_ENC'),
        (F.col('WEIGHT_LBS') * 0.453592 / F.pow(F.col('HEIGHT_CMS') / 100, 2)).alias('BMI'),
        ((F.col('SYSTOLIC_BP') + 2 * F.col('DIASTOLIC_BP')) / 3).alias('MEAN_ARTERIAL_PRESSURE'),
        (F.col('PULSE') / F.col('SYSTOLIC_BP')).alias('SHOCK_INDEX'),
        (F.col('ABS_NEUTROPHIL') / F.col('ABS_LYMPHOCYTE')).alias('NEUTROPHIL_LYMPH_RATIO'),
        F.iff(F.col('PULSE_OXIMETRY') < 92, 1, 0).alias('HYPOXIA_FLAG'),
        (F.datediff('hour', F.col('ENTRY_DATE'), F.col('DISCHARGE_DATE')) / 24.0).alias('LOS_DAYS'),
        F.datediff('hour', F.col('ENTRY_DATE'), F.col('DISCHARGE_DATE')).alias('ADMISSION_DURATION_HRS'),
        F.dayofweek('ENTRY_DATE').alias('ENTRY_DOW'), F.month('ENTRY_DATE').alias('ENTRY_MONTH'))
    return out.select('PRIMARYMRN', 'ENCOUNTER_TS',
                      *[F.col(c).cast('double').alias(c) for c in out.columns if c not in ('PRIMARYMRN', 'ENCOUNTER_TS')])

feat_check = build_codeblue_features(src, medians, gender_map)
assert feat_check.columns == feat_df.columns, 'column mismatch vs feature view logic'
mismatched = feat_check.subtract(feat_df).count()
print(f'pipeline rows: {feat_check.count()} | columns: {len(feat_check.columns)} | rows differing from feature view logic: {mismatched}')

In [ ]:
transforms = {
    'GENDER_ENC': ('GENDER', 'LabelEncoder mapping ' + str(gender_map) + ', unknown = -1'),
    'BMI': ('WEIGHT_LBS, HEIGHT_CMS', 'weight_lbs x 0.453592 / (height_cm/100)^2'),
    'MEAN_ARTERIAL_PRESSURE': ('SYSTOLIC_BP, DIASTOLIC_BP', '(SBP + 2 x DBP) / 3'),
    'SHOCK_INDEX': ('PULSE, SYSTOLIC_BP', 'PULSE / SBP'),
    'NEUTROPHIL_LYMPH_RATIO': ('ABS_NEUTROPHIL, ABS_LYMPHOCYTE', 'ANC / ALC'),
    'HYPOXIA_FLAG': ('PULSE_OXIMETRY', '1 if SpO2 < 92 else 0'),
    'LOS_DAYS': ('ENTRY_DATE, DISCHARGE_DATE', 'hours between / 24 (known only at discharge - leakage risk)'),
    'ADMISSION_DURATION_HRS': ('ENTRY_DATE, DISCHARGE_DATE', 'hours between (known only at discharge - leakage risk)'),
    'ENTRY_DOW': ('ENTRY_DATE', 'day of week'), 'ENTRY_MONTH': ('ENTRY_DATE', 'month'),
}
rows = [(c, c, f'median imputation (fill = {medians[c]:.2f}), cast double', 'Raw') for c in num_cols]
rows += [(f, s, t, 'Derived') for f, (s, t) in transforms.items()]
feature_transform_doc = pd.DataFrame(rows, columns=['FEATURE', 'SOURCE_COLUMNS', 'TRANSFORMATION', 'TYPE'])

training_feature_list = pd.DataFrame({'FEATURE': FEATURE_COLS, 'DTYPE': [str(train_pdf[c].dtype) for c in FEATURE_COLS]})
training_feature_list = training_feature_list.merge(feature_transform_doc[['FEATURE', 'TYPE']], on='FEATURE', how='left')
print(f'{len(FEATURE_COLS)} training features | feature view: {DB}.{FEATURES}.{FV_NAME}${FV_VERSION}')
display(feature_transform_doc)
training_feature_list

## Phase 3 - AutoML-style training & evaluation
Leakage-free, cross-validated model selection:
- **Leakage removed:** `LOS_DAYS` and `ADMISSION_DURATION_HRS` (known only at discharge, and synthetic LOS encodes the label) are excluded from `FEATURE_COLS`. They remain in the feature view but are not model inputs.
- **80/20 stratified split** (seed 42). The 200-row test set is held out and used **only once** for final reporting.
- **Per-family RandomizedSearchCV** (20 candidates x stratified 5-fold CV, ROC-AUC) on the training set for XGBoost (incl. `min_child_weight`, `colsample_bytree`, `reg_lambda`, `reg_alpha`, `gamma`), LightGBM, RandomForest, LogisticRegression (StandardScaler pipeline) and sklearn GradientBoostingClassifier.
- **Class imbalance:** `scale_pos_weight` / `class_weight=balanced` / balanced sample weights.
- **Champion = highest mean CV ROC-AUC**; CV std shows whether families are statistically tied. PR-AUC (average precision) reported alongside ROC-AUC.
- Each family's best candidate is logged (params, CV and test metrics) to experiment `CODE_BLUE_EXPERIMENT`.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from snowflake.ml.experiment import ExperimentTracking

# Drop features only known at discharge (they leak the label in this synthetic data)
LEAKAGE_COLS = ['LOS_DAYS', 'ADMISSION_DURATION_HRS']
FEATURE_COLS = [c for c in FEATURE_COLS if c not in LEAKAGE_COLS]

X = train_pdf[FEATURE_COLS].astype('float64'); y = train_pdf['CODEBLUE'].astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
N_ITER = 20

search_space = {
 'XGBOOST': (XGBClassifier(scale_pos_weight=spw, random_state=SEED, eval_metric='logloss', n_jobs=1), {
    'n_estimators': [100, 200, 300, 500], 'learning_rate': [0.01, 0.03, 0.05, 0.1], 'max_depth': [2, 3, 4, 5],
    'subsample': [0.6, 0.7, 0.8, 1.0], 'colsample_bytree': [0.6, 0.7, 0.8, 0.9], 'min_child_weight': [1, 5, 10],
    'reg_lambda': [1, 3, 5, 10], 'reg_alpha': [0, 0.1, 0.5, 1], 'gamma': [0, 0.1, 0.5, 1]}),
 'LIGHTGBM': (LGBMClassifier(class_weight='balanced', subsample_freq=1, random_state=SEED, verbose=-1, n_jobs=1), {
    'n_estimators': [100, 200, 300, 500], 'learning_rate': [0.01, 0.03, 0.05, 0.1], 'max_depth': [2, 3, 4, 5, -1],
    'num_leaves': [7, 15, 31], 'min_child_samples': [5, 10, 20, 40], 'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9], 'reg_lambda': [0, 1, 5, 10]}),
 'RANDOMFOREST': (RandomForestClassifier(class_weight='balanced_subsample', random_state=SEED, n_jobs=1), {
    'n_estimators': [200, 300, 500], 'max_depth': [4, 6, 8, 12, None], 'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5, 10], 'max_features': ['sqrt', 'log2', 0.5]}),
 'LOGISTICREG': (Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=5000))]), {
    'clf__C': [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}),
 'GRADIENTBOOSTING': (GradientBoostingClassifier(random_state=SEED), {
    'n_estimators': [100, 150, 200, 300], 'learning_rate': [0.01, 0.03, 0.05, 0.1], 'max_depth': [2, 3, 4],
    'subsample': [0.6, 0.7, 0.8, 1.0], 'min_samples_split': [5, 10, 20, 40]}),
}

exp = ExperimentTracking(session=session, database_name=DB, schema_name=SCHEMA)
exp.set_experiment(EXPERIMENT)
stamp = pd.Timestamp.now().strftime('%m%d_%H%M%S')
results, models, cv_tables = [], {}, {}
for fam, (base, dist) in search_space.items():
    n_cand = int(np.prod([len(v) for v in dist.values()]))
    rs = RandomizedSearchCV(base, dist, n_iter=min(N_ITER, n_cand), scoring={'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'},
                            refit='roc_auc', cv=cv, random_state=SEED, n_jobs=-1)
    fit_kw = {'sample_weight': compute_sample_weight('balanced', y_tr)} if fam == 'GRADIENTBOOSTING' else {}
    rs.fit(X_tr, y_tr, **fit_kw)
    i = rs.best_index_; cvr = rs.cv_results_
    cv_tables[fam] = pd.DataFrame(cvr)[['params', 'mean_test_roc_auc', 'std_test_roc_auc', 'mean_test_pr_auc']].sort_values('mean_test_roc_auc', ascending=False)
    m = rs.best_estimator_; p = {k: (v if isinstance(v, (int, float, str, type(None))) else str(v)) for k, v in rs.best_params_.items()}

    # Held-out test set: reporting only, not used for selection
    proba = m.predict_proba(X_te)[:, 1]; pred = (proba >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
    met = dict(cv_roc_auc_mean=float(cvr['mean_test_roc_auc'][i]), cv_roc_auc_std=float(cvr['std_test_roc_auc'][i]),
               cv_pr_auc=float(cvr['mean_test_pr_auc'][i]),
               accuracy=accuracy_score(y_te, pred), precision=precision_score(y_te, pred, zero_division=0),
               recall=recall_score(y_te, pred), f1=f1_score(y_te, pred), roc_auc=roc_auc_score(y_te, proba),
               pr_auc=average_precision_score(y_te, proba), tn=float(tn), fp=float(fp), fn=float(fn), tp=float(tp))
    run = f'{fam}_CV_{stamp}'
    with exp.start_run(run):
        exp.log_params({**p, 'family': fam, 'seed': SEED, 'test_size': 0.2, 'cv_folds': 5, 'n_iter': min(N_ITER, n_cand),
                        'excluded_features': ','.join(LEAKAGE_COLS)})
        exp.log_metrics(met)
    results.append({'run': run, 'family': fam, 'params': p, **met}); models[run] = m
    print(f'{fam:<17} CV ROC-AUC {met["cv_roc_auc_mean"]:.3f} +/- {met["cv_roc_auc_std"]:.3f} | test ROC-AUC {met["roc_auc"]:.3f} | test PR-AUC {met["pr_auc"]:.3f}')

# Champion selected on CV only
leaderboard = pd.DataFrame(results).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
best = leaderboard.iloc[0]; best_model = models[best['run']]
gap = leaderboard.cv_roc_auc_mean.iloc[0] - leaderboard.cv_roc_auc_mean.iloc[1]
print('\n' + leaderboard[['run', 'cv_roc_auc_mean', 'cv_roc_auc_std', 'cv_pr_auc', 'roc_auc', 'pr_auc', 'recall', 'precision', 'f1', 'tn', 'fp', 'fn', 'tp']].round(3).to_string(index=False))
print(f'\nCHAMPION: {best["run"]} {best["params"]}')
print(f'Lead over #2 in CV ROC-AUC: {gap:.3f} (champion CV std {best["cv_roc_auc_std"]:.3f})'
      + (' -> statistically tied' if gap < best['cv_roc_auc_std'] else ''))
print('features used:', len(FEATURE_COLS), '| excluded:', LEAKAGE_COLS)

### Phase 3 Deliverables
| Deliverable | Where |
|---|---|
| Best-performing model | `best_model` / `champion_summary` (selected by mean 5-fold CV ROC-AUC) |
| Hyperparameter tuning summary | `tuning_summary` (per family, 20-candidate randomized CV search) + `cv_tables[family]` (all candidates) + `leaderboard`; also in experiment `CODE_BLUE_EXPERIMENT` |
| Model evaluation report | ROC / PR curves, confusion matrix, classification report on the held-out test set (cell below) |
| Feature importance analysis | `feature_importance_df` + chart |

> Leakage features `LOS_DAYS` and `ADMISSION_DURATION_HRS` are excluded from training. Test metrics are from a held-out set not used for selection; with only 19 test positives, differences of ~0.02 ROC-AUC between families are within noise.

In [ ]:
champion_summary = pd.Series({'run': best['run'], 'algorithm': best['family'], 'params': best['params'],
    **{k: round(float(best[k]), 4) for k in ['cv_roc_auc_mean', 'cv_roc_auc_std', 'cv_pr_auc', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']}})
tuning_summary = pd.DataFrame([{
    'family': fam, 'candidates': len(t),
    'best_cv_roc_auc': t.mean_test_roc_auc.iloc[0], 'cv_std': t.std_test_roc_auc.iloc[0],
    'worst_cv_roc_auc': t.mean_test_roc_auc.iloc[-1], 'best_cv_pr_auc': t.mean_test_pr_auc.max(),
    'test_roc_auc': float(leaderboard.loc[leaderboard.family == fam, 'roc_auc'].iloc[0]),
    'test_recall': float(leaderboard.loc[leaderboard.family == fam, 'recall'].iloc[0]),
    'best_params': str(t.params.iloc[0])} for fam, t in cv_tables.items()]
).sort_values('best_cv_roc_auc', ascending=False).round(4).reset_index(drop=True)
print(champion_summary.to_string())
tuning_summary

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, classification_report, ConfusionMatrixDisplay

te_proba = best_model.predict_proba(X_te)[:, 1]
te_pred = (te_proba >= 0.5).astype(int)
fpr, tpr, _ = roc_curve(y_te, te_proba)
prec, rec, _ = precision_recall_curve(y_te, te_proba)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_te, te_proba):.3f}'); ax[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
ax[0].set(title='ROC curve', xlabel='False positive rate', ylabel='True positive rate'); ax[0].legend()
ax[1].plot(rec, prec, label=f'AP = {average_precision_score(y_te, te_proba):.3f}'); ax[1].axhline(y_te.mean(), color='k', ls='--', lw=0.8)
ax[1].set(title='Precision-recall curve', xlabel='Recall', ylabel='Precision'); ax[1].legend()
ConfusionMatrixDisplay.from_predictions(y_te, te_pred, display_labels=['No Code Blue', 'Code Blue'], ax=ax[2], colorbar=False)
ax[2].set_title('Confusion matrix (threshold 0.5)')
fig.suptitle(f'Champion: {best["run"]}'); plt.tight_layout(); plt.show()
print(classification_report(y_te, te_pred, target_names=['No Code Blue', 'Code Blue'], digits=3))

In [ ]:
_est = best_model.named_steps['clf'] if hasattr(best_model, 'named_steps') else best_model
_imp = _est.feature_importances_ if hasattr(_est, 'feature_importances_') else np.abs(_est.coef_[0])
feature_importance_df = (pd.DataFrame({'FEATURE': FEATURE_COLS, 'IMPORTANCE': _imp / _imp.sum()})
    .sort_values('IMPORTANCE', ascending=False).reset_index(drop=True))
feature_importance_df['CUMULATIVE'] = feature_importance_df['IMPORTANCE'].cumsum()
top = feature_importance_df.head(15).iloc[::-1]
plt.figure(figsize=(8, 6)); plt.barh(top.FEATURE, top.IMPORTANCE)
plt.title(f'Top 15 feature importances ({best["family"]})'); plt.xlabel('Normalized importance'); plt.tight_layout(); plt.show()
feature_importance_df.round(4)

## Phase 4 - Model Registry deployment
Registers the champion as CODEBLUE_MODEL version V1_0 (warehouse target) with explainability enabled (SHAP explain function). Stored in version metadata: training metadata, performance metrics, model parameters, feature definitions and feature importance. V1_0 is set as the default (approved) version.

In [ ]:
from snowflake.ml.registry import Registry
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)
try:
    reg.delete_model(MODEL_NAME)   # idempotent re-run inside DEMO_DB
except Exception:
    pass

est = best_model.named_steps['clf'] if hasattr(best_model, 'named_steps') else best_model
imp = est.feature_importances_ if hasattr(est, 'feature_importances_') else np.abs(est.coef_[0])
feature_importance = dict(sorted({f: float(v) for f, v in zip(FEATURE_COLS, imp / imp.sum())}.items(), key=lambda kv: -kv[1]))
feature_definitions = {'source_feature_view': f'{DB}.{FEATURES}.{FV_NAME}${FV_VERSION}', 'features': FEATURE_COLS,
    'imputation': 'median', 'medians': {k: float(v) for k, v in medians.items()}, 'gender_label_encoding': gender_map}
metrics = {**{k: float(best[k]) for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']},
    'confusion_matrix': {'tn': int(best.tn), 'fp': int(best.fp), 'fn': int(best.fn), 'tp': int(best.tp)},
    'model_params': {k: (v if isinstance(v, (int, float, str, type(None))) else str(v)) for k, v in best['params'].items()},
    'algorithm': best['family'], 'experiment': EXPERIMENT, 'run': best['run'],
    'training_metadata': {'train_rows': int(len(X_tr)), 'test_rows': int(len(X_te)), 'positive_rate': float(y.mean()), 'seed': SEED, 'trained_at': str(pd.Timestamp.now())},
    'feature_definitions': feature_definitions, 'feature_importance': feature_importance}

log_kwargs = dict(model_name=MODEL_NAME, version_name=MODEL_VERSION, sample_input_data=X_tr.head(100),
    target_platforms=['WAREHOUSE'], metrics=metrics, comment=f'Code Blue risk model ({best["family"]}) trained on {FV_NAME}/{FV_VERSION}')
try:
    mv = reg.log_model(best_model, options={'enable_explainability': True}, **log_kwargs)
    EXPLAIN_ENABLED = True
except Exception as e:
    print('explainability unavailable, registering without it:', str(e)[:200])
    mv = reg.log_model(best_model, options={'enable_explainability': False}, **log_kwargs)
    EXPLAIN_ENABLED = False
reg.get_model(MODEL_NAME).default = MODEL_VERSION
print('Registered', FQ_MODEL, mv.version_name, '| explain:', EXPLAIN_ENABLED)
print([f['name'] for f in mv.show_functions()])
print('Top features:', list(feature_importance.items())[:8])

### Phase 4 Deliverables
| Deliverable | Where |
|---|---|
| Registered production-ready model | `DEMO_DB.CLINIC.CODEBLUE_MODEL` version `V1_0` (default version) |
| Model Registry entry | `model_versions` (SHOW VERSIONS) |
| Model metadata documentation | `model_metadata_doc` (cell below) |

In [ ]:
%%sql -r model_versions
SHOW VERSIONS IN MODEL DEMO_DB.CLINIC.CODEBLUE_MODEL;

In [ ]:
reg_metrics = mv.show_metrics()
model_metadata_doc = pd.DataFrame([
    ('Model', FQ_MODEL), ('Version', mv.version_name),
    ('Default version', reg.get_model(MODEL_NAME).default.version_name),
    ('Algorithm', reg_metrics.get('algorithm')), ('Experiment run', f"{reg_metrics.get('experiment')} / {reg_metrics.get('run')}"),
    ('Comment', mv.comment), ('Target platform', 'WAREHOUSE'), ('Explainability enabled', EXPLAIN_ENABLED),
    ('Functions', ', '.join(f['name'] for f in mv.show_functions())),
    ('Source feature view', feature_definitions['source_feature_view']), ('Feature count', len(FEATURE_COLS)),
    ('Training rows / test rows', f"{reg_metrics['training_metadata']['train_rows']} / {reg_metrics['training_metadata']['test_rows']}"),
    ('Positive rate', round(reg_metrics['training_metadata']['positive_rate'], 4)),
    ('Trained at', reg_metrics['training_metadata']['trained_at']),
    ('Hyperparameters', reg_metrics.get('model_params')),
    *[(f'Test {k}', round(reg_metrics[k], 4)) for k in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']],
    ('Confusion matrix', reg_metrics.get('confusion_matrix')),
], columns=['PROPERTY', 'VALUE'])
model_metadata_doc['VALUE'] = model_metadata_doc['VALUE'].astype(str)
model_metadata_doc

## Phase 5 - Production inference
Loads the default (approved) CODEBLUE_MODEL version, retrieves features for all 1,000 encounters from the same feature view (identical pipeline), and scores with predict_proba (threshold 0.5) plus per-row SHAP explanations (top 5 drivers with direction). Results go to CODEBLUE_SCORING_RESULTS.

In [ ]:
prod_mv = Registry(session=session, database_name=DB, schema_name=SCHEMA).get_model(MODEL_NAME).default
score_spine = session.table(f'{DB}.{SCHEMA}.PATIENT_CODEBLUE').select('PRIMARYMRN', F.col('ENTRY_DATE').alias('ENCOUNTER_TS'))
inf_pdf = fs.retrieve_feature_values(spine_df=score_spine, features=[fv], spine_timestamp_col='ENCOUNTER_TS').to_pandas().reset_index(drop=True)
Xs = inf_pdf[FEATURE_COLS].astype('float64')

proba_out = prod_mv.run(Xs, function_name='predict_proba')
out_cols = [c for c in proba_out.columns if c not in FEATURE_COLS]
p1 = proba_out[out_cols[-1]].astype(float).values

try:
    ex = prod_mv.run(Xs, function_name='explain')
    ex_cols = [c for c in ex.columns if c not in FEATURE_COLS]
    shap = ex[ex_cols].astype(float).values
    ex_feats = [c.replace('_explanation', '').replace('_EXPLANATION', '') for c in ex_cols]
    method = 'SHAP'
except Exception as e:
    print('explain failed, using importance x z-score fallback:', str(e)[:150])
    z = ((Xs - X_tr.mean()) / X_tr.std().replace(0, 1)).values
    shap = z * np.array([feature_importance[f] for f in FEATURE_COLS]); ex_feats = FEATURE_COLS; method = 'IMPORTANCE_X_ZSCORE'

def rationale(i):
    order = np.argsort(-np.abs(shap[i]))[:5]
    drivers = [{'feature': ex_feats[j], 'value': round(float(Xs.iloc[i][ex_feats[j]]), 3) if ex_feats[j] in Xs else None,
                'contribution': round(float(shap[i][j]), 4), 'direction': 'increases risk' if shap[i][j] > 0 else 'decreases risk'} for j in order]
    return json.dumps({'method': method, 'top_drivers': drivers})

scoring = pd.DataFrame({'PRIMARYMRN': inf_pdf['PRIMARYMRN'], 'CODEBLUE_PREDICTION': (p1 >= 0.5).astype(int),
    'CODEBLUE_PROBABILITY': p1.round(4), 'MODEL_VERSION': f'{MODEL_NAME}/{prod_mv.version_name}',
    'SCORING_TIMESTAMP': pd.Timestamp.now(), 'EXPLAINABILITY_JSON': [rationale(i) for i in range(len(Xs))]})
session.write_pandas(scoring, 'TMP_CODEBLUE_SCORES', database=DB, schema=SCHEMA, auto_create_table=True, overwrite=True, use_logical_type=True)
session.sql(f'''CREATE OR REPLACE TABLE {DB}.{SCHEMA}.CODEBLUE_SCORING_RESULTS AS
SELECT PRIMARYMRN, CODEBLUE_PREDICTION, CODEBLUE_PROBABILITY, MODEL_VERSION, SCORING_TIMESTAMP::TIMESTAMP_NTZ AS SCORING_TIMESTAMP,
       PARSE_JSON(EXPLAINABILITY_JSON) AS EXPLAINABILITY_ATTRIBUTES FROM {DB}.{SCHEMA}.TMP_CODEBLUE_SCORES''').collect()
session.sql(f'DROP TABLE IF EXISTS {DB}.{SCHEMA}.TMP_CODEBLUE_SCORES').collect()
print('scored rows:', session.table(f'{DB}.{SCHEMA}.CODEBLUE_SCORING_RESULTS').count(), '| predicted positives:', int(scoring.CODEBLUE_PREDICTION.sum()))

### Phase 5 Deliverables
| Deliverable | Where |
|---|---|
| CODEBLUE_SCORING_RESULTS table | `DEMO_DB.CLINIC.CODEBLUE_SCORING_RESULTS` (created above) |
| Prediction summary report | `prediction_summary` |
| Risk distribution analysis | `risk_distribution` (bands: Low < 0.3, Medium 0.3-0.7, High >= 0.7) + histogram |

In [ ]:
%%sql -r prediction_summary
SELECT ANY_VALUE(MODEL_VERSION) AS MODEL_VERSION, MAX(SCORING_TIMESTAMP) AS SCORED_AT,
       COUNT(*) AS SCORED_PATIENTS, SUM(CODEBLUE_PREDICTION) AS PREDICTED_CODEBLUE,
       ROUND(AVG(CODEBLUE_PREDICTION) * 100, 2) AS PREDICTED_POSITIVE_PCT,
       ROUND(AVG(CODEBLUE_PROBABILITY), 4) AS AVG_PROBABILITY, ROUND(MEDIAN(CODEBLUE_PROBABILITY), 4) AS MEDIAN_PROBABILITY,
       ROUND(MAX(CODEBLUE_PROBABILITY), 4) AS MAX_PROBABILITY
FROM DEMO_DB.CLINIC.CODEBLUE_SCORING_RESULTS;

In [ ]:
%%sql -r risk_distribution
SELECT CASE WHEN CODEBLUE_PROBABILITY >= 0.7 THEN '3-High' WHEN CODEBLUE_PROBABILITY >= 0.3 THEN '2-Medium' ELSE '1-Low' END AS RISK_BAND,
       COUNT(*) AS PATIENTS, ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 1) AS PCT_OF_PATIENTS,
       ROUND(AVG(CODEBLUE_PROBABILITY), 4) AS AVG_PROBABILITY, SUM(CODEBLUE_PREDICTION) AS PREDICTED_CODEBLUE
FROM DEMO_DB.CLINIC.CODEBLUE_SCORING_RESULTS
GROUP BY 1 ORDER BY 1;

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
plt.hist(p1, bins=40, color='steelblue', edgecolor='white')
for t, lbl in [(0.3, 'Medium'), (0.7, 'High')]:
    plt.axvline(t, color='red', ls='--', lw=1); plt.text(t + 0.01, plt.ylim()[1] * 0.9, lbl, color='red')
plt.yscale('log'); plt.title('Code Blue probability distribution (log count)')
plt.xlabel('Predicted probability'); plt.ylabel('Patients'); plt.tight_layout(); plt.show()

## Phase 6 - Final business-ready dataset
LEFT JOIN of PATIENT_CODEBLUE with CODEBLUE_SCORING_RESULTS on PRIMARYMRN (one row per encounter), retaining all source fields and appending prediction, probability, model version, scoring timestamp and explainability attributes.

In [ ]:
%%sql -r final_create
CREATE OR REPLACE TABLE DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS AS
SELECT p.*, s.CODEBLUE_PREDICTION, s.CODEBLUE_PROBABILITY, s.MODEL_VERSION, s.SCORING_TIMESTAMP, s.EXPLAINABILITY_ATTRIBUTES
FROM DEMO_DB.CLINIC.PATIENT_CODEBLUE p
LEFT JOIN DEMO_DB.CLINIC.CODEBLUE_SCORING_RESULTS s ON p.PRIMARYMRN = s.PRIMARYMRN;

In [ ]:
%%sql -r final_summary
SELECT COUNT(*) AS TOTAL_RECORDS, COUNT(CODEBLUE_PROBABILITY) AS SCORED, SUM(CODEBLUE) AS ACTUAL_CODEBLUE,
       SUM(CODEBLUE_PREDICTION) AS PREDICTED_CODEBLUE, SUM(IFF(CODEBLUE=1 AND CODEBLUE_PREDICTION=1,1,0)) AS TRUE_POSITIVES,
       ROUND(AVG(DATEDIFF('hour', ENTRY_DATE, DISCHARGE_DATE))/24,1) AS AVG_LOS_DAYS,
       ROUND(AVG(IFF(CODEBLUE=1, DATEDIFF('hour', ENTRY_DATE, DISCHARGE_DATE)/24, NULL)),1) AS AVG_LOS_CODEBLUE,
       COUNT_IF(CODEBLUE_PROBABILITY >= 0.7) AS HIGH_RISK_PATIENTS, ANY_VALUE(MODEL_VERSION) AS MODEL_VERSION
FROM DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS;

### Phase 6 Deliverables
| Deliverable | Where |
|---|---|
| Fully enriched patient prediction dataset | `DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS` (all source columns + predictions) |
| Prediction probability scores | `CODEBLUE_PROBABILITY` / `CODEBLUE_PREDICTION` columns (`enriched_sample`) |
| Explainability insights | `explainability_insights` (top risk drivers among high-risk patients) |
| Business-ready analytical dataset | view `DEMO_DB.CLINIC.CODEBLUE_PATIENT_RISK_V` (`business_ready_sample`) |

In [ ]:
%%sql -r enriched_sample
SELECT PRIMARYMRN, AGE, GENDER, CODEBLUE, CODEBLUE_PREDICTION, CODEBLUE_PROBABILITY, MODEL_VERSION, SCORING_TIMESTAMP
FROM DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS
ORDER BY CODEBLUE_PROBABILITY DESC
LIMIT 20;

In [ ]:
%%sql -r explainability_insights
SELECT d.value:feature::STRING AS RISK_DRIVER,
       COUNT_IF(d.index = 0) AS TIMES_TOP_DRIVER,
       COUNT(*) AS TIMES_IN_TOP5,
       ROUND(COUNT(*) * 100 / ANY_VALUE(t.HIGH_RISK), 1) AS PCT_HIGH_RISK_PATIENTS,
       ROUND(AVG(d.value:contribution::FLOAT), 4) AS AVG_CONTRIBUTION,
       COUNT_IF(d.value:direction::STRING = 'increases risk') AS TIMES_INCREASES_RISK
FROM DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS r,
     LATERAL FLATTEN(input => r.EXPLAINABILITY_ATTRIBUTES:top_drivers) d,
     (SELECT COUNT_IF(CODEBLUE_PROBABILITY >= 0.7) AS HIGH_RISK FROM DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS) t
WHERE r.CODEBLUE_PROBABILITY >= 0.7
GROUP BY 1
ORDER BY TIMES_TOP_DRIVER DESC, TIMES_IN_TOP5 DESC;

In [ ]:
%%sql -r business_view_create
CREATE OR REPLACE VIEW DEMO_DB.CLINIC.CODEBLUE_PATIENT_RISK_V
COMMENT = 'Business-ready Code Blue risk dataset: one row per patient encounter with risk band and top drivers'
AS
SELECT PRIMARYMRN, AGE, GENDER, ENTRY_DATE AS ADMISSION_TS, DISCHARGE_DATE AS DISCHARGE_TS,
       ROUND(DATEDIFF('hour', ENTRY_DATE, DISCHARGE_DATE) / 24, 1) AS LENGTH_OF_STAY_DAYS,
       RESPIRATION, SYSTOLIC_BP, PULSE, PULSE_OXIMETRY, TEMPERATURE, BNP, ALBUMIN, BLOOD_CULTURE,
       CODEBLUE AS ACTUAL_CODEBLUE, CODEBLUE_PREDICTION AS PREDICTED_CODEBLUE,
       ROUND(CODEBLUE_PROBABILITY * 100, 1) AS RISK_SCORE_PCT,
       CASE WHEN CODEBLUE_PROBABILITY >= 0.7 THEN 'High' WHEN CODEBLUE_PROBABILITY > 0.3 THEN 'Medium' ELSE 'Low' END AS RISK_BAND,
       EXPLAINABILITY_ATTRIBUTES:top_drivers[0]:feature::STRING AS TOP_RISK_DRIVER_1,
       EXPLAINABILITY_ATTRIBUTES:top_drivers[1]:feature::STRING AS TOP_RISK_DRIVER_2,
       EXPLAINABILITY_ATTRIBUTES:top_drivers[2]:feature::STRING AS TOP_RISK_DRIVER_3,
       EXPLAINABILITY_ATTRIBUTES:method::STRING AS EXPLANATION_METHOD,
       MODEL_VERSION, SCORING_TIMESTAMP
FROM DEMO_DB.CLINIC.CODEBLUE_PREDICTION_RESULTS;

In [ ]:
%%sql -r business_ready_sample
SELECT RISK_BAND, COUNT(*) AS PATIENTS, ROUND(AVG(RISK_SCORE_PCT), 1) AS AVG_RISK_SCORE_PCT,
       SUM(ACTUAL_CODEBLUE) AS ACTUAL_CODEBLUE, SUM(PREDICTED_CODEBLUE) AS PREDICTED_CODEBLUE,
       MODE(TOP_RISK_DRIVER_1) AS MOST_COMMON_TOP_DRIVER
FROM DEMO_DB.CLINIC.CODEBLUE_PATIENT_RISK_V
GROUP BY 1 ORDER BY AVG_RISK_SCORE_PCT DESC;